# 패션 특화 임베딩 모델 3종 비교 검증 노트북

**스파이크/검증용 코드입니다. 프로덕션 코드가 아닙니다.**

상품 정보 분류(카테고리+색상)에 쓸 임베딩 체크포인트를 고르기 위해, **패션 도메인으로
파인튜닝된 모델 3종**을 범용 CLIP 대조군과 같은 조건에서 비교한다.

| 키 | 체크포인트 | 고른 이유 |
| --- | --- | --- |
| `clip_vit_b32` | `openai/clip-vit-base-patch32` | **범용 대조군.** 지금까지 모든 실측의 기준선(`REPORT.md` §V2a~§17) |
| `fashion_clip` | `patrickjohncyh/fashion-clip` | FashionCLIP 2.0. LAION CLIP ViT-B/32 + 패션 상품 80만 건 파인튜닝 |
| `marqo_fashion_clip` | `Marqo/marqo-fashionCLIP` | GCL(카테고리·색상·소재를 학습 신호로 사용). FashionCLIP 2.0 대비 recall@1 +22% |
| `marqo_fashion_siglip` | `Marqo/marqo-fashionSigLIP` | 같은 레시피의 SigLIP 판, +57%. 공개된 패션 임베딩 중 보고 성능 최고 |

`zooclaw_fashion_siglip2`는 레지스트리에는 등록돼 있지만 이번 비교에서는 제외했다
(§2에 주석으로 남겨뒀다 — 켜고 싶으면 한 줄 주석만 풀면 된다).

## 세 가지 테스트

| # | 테스트 | 어떻게 재는가 |
| --- | --- | --- |
| 1 | **중분류(소분류) 30-way 추론** | `evaluate_category(mode="one_stage")` top-1 / top-3 |
| 2 | **중분류 → 대분류 역산** | 테스트 1의 예측을 `taxonomy.major_of()`로 접는다. **추가 추론 없음** |
| 3 | **색상 추론** | `evaluate_color()` primary 정확도. 모델당 원본 / 화이트밸런스(`_wb`) 2조건 |

여기에 지연 실측(§5)을 더해 §6 스코어보드에서 모델당 한 줄로 정리한다.

테스트 1과 2가 추론을 공유하는 게 중요하다 — `CategoryPrediction`에 예측 소분류와 정답
대분류가 같이 들어 있어서, 역산 정확도는 이미 나온 결과를 접기만 하면 된다. 그래서 모델당
추론 패스는 **카테고리 1회 + 색상 2회**로 끝난다.

## 왜 기존 노트북과 분리했나

`major_category_and_color_check.ipynb`는 축이 "대분류 6종 + 색상 개선 기법 + VLM 대조군"
이고 Qwen3-VL 로드가 전제라 GPU가 필요하다. 이 노트북의 질문은 "패션 모델 중 무엇을
쓸 것인가"라는 **모델 × 과제 매트릭스**이고 CLIP 계열만 쓰므로 VLM 없이, CPU로도 돈다.
그리고 이쪽은 로직을 인라인하지 않고 `evaluate.py`를 그대로 재사용해서 top-3·혼동행렬·
대분류별 정확도·ECE·임계값 추천을 공짜로 얻는다.

## 실행 전 준비

1. `pip install open_clip_torch` — **Marqo 두 모델은 `open_clip`으로 싣는다.** 모델 카드가
   함께 제시하는 `AutoModel(trust_remote_code=True)` 경로는 **transformers 5.x에서 두 모델
   모두 로드가 실패한다**(실측 5.14.1: fashionCLIP은 `Unrecognized image processor`,
   fashionSigLIP은 `Cannot copy out of meta tensor`). 자세한 근거는
   `classify.CATEGORY_MODEL_LOAD_OPTIONS` 주석 참고. 없으면 **그 두 개만** 스킵되고
   나머지는 그대로 돈다(§0에서 먼저 확인한다).
2. 체크포인트 최초 다운로드 약 1.5GB.
3. 위에서부터 순서대로 실행. §5(지연)만 `RUN_LATENCY` 플래그로 끌 수 있다.

## 주의 — "정확도 0.00"은 측정값이 아닐 수 있다

`evaluate_category`/`evaluate_color`는 샘플 단위로 예외를 삼키므로(실패 격리), **백엔드
로딩이 통째로 깨져도** 예외 대신 전 행 오답인 "0.00"이 돌아온다. §3·§4는
`evaluate.all_failed()`로 그런 런을 걸러 `[SKIP]`으로 표시하고 표에 `NaN`을 넣는다 —
**0.00과 NaN은 뜻이 다르다**(전자는 실측, 후자는 못 돌았음). 일부 행만 실패한 경우는
`error_n` 컬럼에 몇 행인지 나온다(그 행들은 오답으로 세어진 것이다).

## 0. 실행 전 프리플라이트

체크포인트를 1.5GB 받아놓고 로드 단계에서 의존성 때문에 실패하는 걸 막는다. 여기서
`open_clip 없음`이 뜨면 Marqo 2종은 §3·§4에서 스킵된다(나머지 비교는 그대로 성립).

In [ ]:
import importlib.util
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

print(f"python: {sys.version.split()[0]}")

try:
    import torch

    print(f"torch: {torch.__version__} (cuda={torch.cuda.is_available()})")
except ImportError:
    print("torch 없음 — validation/requirements-test.txt를 먼저 설치할 것")

if importlib.util.find_spec("open_clip") is None:
    print("open_clip 없음 -> marqo_fashion_* 2종 스킵됨. `pip install open_clip_torch`")
else:
    print("open_clip 설치됨 — Marqo 후보 실행 가능")

# 이미 받아둔 체크포인트 확인 — 비어 있으면 첫 실행에서 약 1.5GB를 내려받는다.
_hub = Path.home() / ".cache" / "huggingface" / "hub"
_cached = sorted(d.name for d in _hub.glob("models--*")) if _hub.exists() else []
print(f"\nHF 캐시({_hub}): {len(_cached)}개")
for _name in _cached:
    print(f"  {_name}")

## 1. 정답셋 로드

`product_info_check.ipynb`·`major_category_and_color_check.ipynb`와 **같은**
`data/ground_truth.json`을 쓴다. `evaluate_category()`/`evaluate_color()`가 `synthetic`
행을 자동으로 건너뛰므로 이 노트북의 모든 지표는 같은 N 위에 선다.

In [ ]:
import pandas as pd

from dataset import load_ground_truth, summarize
from taxonomy import major_of

gt = load_ground_truth()
real_samples = [s for s in gt.samples if not s.synthetic]
real_images = [s.image_path for s in real_samples]

print(summarize(gt))
print(f"\n실제 사진 {len(real_samples)}장 — 아래 모든 지표의 N이다.")

_missing = [s.image_file for s in real_samples if not s.image_path.exists()]
print(f"이미지 파일 누락: {_missing if _missing else '없음'}")

## 2. 설정 상수

스왑 가능한 값은 전부 이 셀에 모은다(기존 노트북 관례). **비교 축은 체크포인트 하나로
고정한다** — 프롬프트 템플릿·분류 모드·정답셋은 전 모델 공통이다. 여기서 프롬프트까지
같이 흔들면 "패션 파인튜닝 덕분인지 프롬프트 덕분인지"를 못 가른다.

In [ ]:
from classify import CATEGORY_BACKENDS
from color import COLOR_BACKENDS

# baseline이 목록 첫 줄인 게 중요하다 — "패션 파인튜닝이 범용 대비 얼마나 움직였나"가
# 이 노트북의 질문이라, 대조군 없이는 어떤 숫자도 해석되지 않는다.
BASELINE = "clip_vit_b32"
MODELS = [
    BASELINE,
    "fashion_clip",
    "marqo_fashion_clip",
    "marqo_fashion_siglip",
    # "zooclaw_fashion_siglip2",  # 레지스트리엔 있지만 이번 비교에서는 제외
]

TEMPLATE_KEY = "en_clip_style"  # taxonomy.PROMPT_TEMPLATES의 키. 소분류 검증에서 최고였던 것
CATEGORY_MODE = "one_stage"  # 소분류 30-way 한 번에 — 역산 대분류도 이 결과에서 계산된다
COLOR_CONDITIONS = ("", "_wb")  # 원본 / 화이트밸런스 전처리(§13·§17에서 유일하게 재현된 개선)
RUN_LATENCY = True  # 이미지 전체를 한 번 더 돌므로 실행 시간이 대략 2배가 된다

# 오타로 반나절 뒤에 KeyError를 보는 대신 여기서 즉시 잡는다.
_unknown_cat = [m for m in MODELS if m not in CATEGORY_BACKENDS]
_unknown_color = [
    f"{m}{c}" for m in MODELS for c in COLOR_CONDITIONS if f"{m}{c}" not in COLOR_BACKENDS
]
assert not _unknown_cat, f"CATEGORY_BACKENDS에 없는 키: {_unknown_cat}"
assert not _unknown_color, f"COLOR_BACKENDS에 없는 키: {_unknown_color}"

print(f"카테고리 백엔드 {len(MODELS)}종 / 색상 백엔드 {len(MODELS) * len(COLOR_CONDITIONS)}종")
print(f"template={TEMPLATE_KEY}, mode={CATEGORY_MODE}, RUN_LATENCY={RUN_LATENCY}")

## 3. 테스트 1·2 — 중분류 30-way 추론 + 대분류 역산

모델별로 `evaluate_category()`를 **한 번씩만** 돌리고 세 지표를 동시에 뽑는다:

- `sub_top1` / `sub_top3` — 테스트 1(중분류 30종)
- `major_inverted` — 테스트 2. 예측 소분류를 `major_of()`로 접어 정답 대분류와 비교

`major_category_and_color_check.ipynb` §2-1에서 범용 CLIP의 역산 대분류는 **1.00**,
소분류 top-1은 `REPORT.md` §8 기준 0.71~0.90이었다. 역산은 이미 천장이라 여기서 볼 건
개선이 아니라 **회귀 여부**(N이 작아 1.00은 쉽게 깨진다)이고, 실제 선택 기준이 되는 건
소분류 정확도 쪽이다.

In [ ]:
from evaluate import all_failed, error_count, evaluate_category


def _failed_row(backend_key, reason, extra_keys):
    print(f"[SKIP] {backend_key}: {reason}")
    row = {"model": backend_key, "n": 0, "error_n": 0, "error": str(reason)[:160]}
    row.update({k: float("nan") for k in extra_keys})
    return row


CATEGORY_METRICS = ("sub_top1", "sub_top3", "major_inverted")

category_preds = {}  # backend -> list[CategoryPrediction]. §3-1~§3-3·§6·§7이 재사용한다
category_rows = []

for backend_key in MODELS:
    try:
        preds = evaluate_category(gt.samples, backend_key, TEMPLATE_KEY, mode=CATEGORY_MODE)
    except Exception as exc:
        # 백엔드 **생성** 실패(등록 안 된 키 등)는 evaluate_category의 샘플 단위 try/except
        # 바깥에서 터진다 — 여기서 받아야 나머지 모델 비교가 살아남는다.
        category_rows.append(
            _failed_row(backend_key, f"{type(exc).__name__}: {exc}", CATEGORY_METRICS)
        )
        continue

    # 백엔드 **로딩** 실패는 예외로 안 올라온다 — evaluate_category가 샘플마다 삼켜서
    # 전 행이 오답이 되고 "정확도 0.00"이라는 정상 결과처럼 보인다(evaluate.all_failed
    # docstring 참고, marqo 백엔드에서 실제로 물렸다). 숫자를 표에 싣기 전에 거른다.
    failure = all_failed(preds)
    if failure:
        category_rows.append(
            _failed_row(backend_key, f"전 샘플 실패 — {failure}", CATEGORY_METRICS)
        )
        continue

    category_preds[backend_key] = preds
    n = len(preds)
    # 테스트 2 — 추가 추론 없이 테스트 1의 예측을 접는다. p.major는 **정답** 대분류다
    # (Sample.major_category). 예측이 None인 행(샘플 단위 실패)은 오답으로 센다.
    inverted = sum(1 for p in preds if p.predicted and major_of(p.predicted) == p.major)
    category_rows.append(
        {
            "model": backend_key,
            "n": n,
            "sub_top1": sum(p.correct_top1 for p in preds) / n,
            "sub_top3": sum(p.correct_top3 for p in preds) / n,
            "major_inverted": inverted / n,
            "error_n": error_count(preds),  # 0이 아니면 그 행들은 오답으로 세어진 것이다
            "error": "",
        }
    )
    _r = category_rows[-1]
    print(
        f"[{backend_key}] 소분류 top1={_r['sub_top1']:.2f} top3={_r['sub_top3']:.2f}"
        f" / 역산 대분류={_r['major_inverted']:.2f}"
        + (f"  (실패 {_r['error_n']}행 포함)" if _r["error_n"] else "")
    )

category_df = pd.DataFrame(category_rows)
category_df

### 3-1. 이미지별 예측 비교표

스코어 한 줄로는 "어느 이미지에서 갈리는지"가 안 보인다. 17장을 행으로, 모델을 열로 펴서
나란히 놓는다. 모델 전부가 똑같이 틀리는 이미지는 **정답셋/이미지 품질 문제**일 가능성이
높고, 특정 모델만 틀리는 이미지가 그 모델의 약점이다.

In [ ]:
if category_preds:
    _base = next(iter(category_preds.values()))
    compare = pd.DataFrame(
        {
            "image_file": [p.image_file for p in _base],
            "truth_sub": [p.truth for p in _base],
            "truth_major": [p.major for p in _base],
        }
    )
    # evaluate_category는 항상 같은 순서(정답셋 순, synthetic 제외)로 돌려주므로 열 병합이 안전하다.
    for backend_key, preds in category_preds.items():
        compare[backend_key] = [p.predicted for p in preds]
    display(compare)

    _cols = list(category_preds)
    disagree = compare[compare[_cols].nunique(axis=1) > 1]
    print(f"모델 간 예측이 갈린 이미지: {len(disagree)}/{len(compare)}건")
    display(disagree)

    unanimous_wrong = compare[
        (compare[_cols].nunique(axis=1) == 1) & (compare[_cols[0]] != compare["truth_sub"])
    ]
    print(f"전 모델이 똑같이 틀린 이미지: {len(unanimous_wrong)}건 (정답셋/이미지 품질 의심)")
    display(unanimous_wrong)
else:
    print("성공한 카테고리 백엔드가 없다 — §3을 먼저 실행할 것.")

### 3-2. 모델별 혼동행렬

`taxonomy.CONFUSION_PAIRS`에 사용자가 지정한 혼동쌍(JACKET/COAT, SLACKS/CARGO_PANTS 등)이
있다. 패션 파인튜닝이 실제로 도움이 됐다면 **그 쌍들에서** 오답이 줄어야 한다 — 전체
정확도가 같아도 어디서 틀리는지가 다르면 의미가 다르다.

In [ ]:
from evaluate import confusion_matrix

# confusion_matrix()는 {정답: {예측: n}} 중첩 dict다. pd.DataFrame(cm)을 그냥 쓰면 바깥
# 키가 **열**이 돼서 행=예측/열=정답으로 뒤집힌다(기존 노트북들의 캡션과 실제가 어긋나 있다).
# 그래서 .T로 되돌려 행=정답으로 맞춘다.
if not category_preds:
    print("성공한 카테고리 백엔드가 없다 — §3을 먼저 실행할 것.")
for backend_key, preds in category_preds.items():
    cm_df = pd.DataFrame(confusion_matrix(preds)).fillna(0).astype(int).T
    cm_df = cm_df.loc[cm_df.sum(axis=1) > 0, cm_df.sum(axis=0) > 0]
    print(f"\n[{backend_key}] 행=정답, 열=예측")
    display(cm_df)

### 3-3. 대분류·촬영유형별 소분류 정확도

정답셋이 상의/하의/아우터에 쏠려 있고 촬영유형도 `flat` 14 / `hanging` 4로 치우쳐 있다.
전체 top-1 하나만 보면 이 쏠림에 가려지는 차이가 있어서 그룹별로도 쪼개 본다.

In [ ]:
from evaluate import accuracy_by

for group_key in ("major", "shot_type"):
    rows = []
    for backend_key, preds in category_preds.items():
        for group, stat in accuracy_by(preds, group_key).items():
            rows.append(
                {
                    "model": backend_key,
                    group_key: group,
                    "n": stat["n"],
                    "top1": stat["top1"],
                }
            )
    if rows:
        pivot = pd.DataFrame(rows).pivot(index=group_key, columns="model", values="top1")
        counts = pd.DataFrame(rows).groupby(group_key)["n"].first()
        print(f"\n{group_key}별 소분류 top1 (n: {counts.to_dict()})")
        display(pivot)

## 4. 테스트 3 — 색상 추론 (모델 × 원본/`_wb`)

색상은 `REPORT.md` §17 기준 CLIP zero-shot 0.41(+화이트밸런스+편향 보정 0.50)에서 막혀
있고, 프롬프트 앙상블(§11)·색상 프롬프트 v2/v3(§12·§15)·스와치 참조(§16)는 전부 회귀했다.
남은 축이 체크포인트라 여기서 확인한다.

**Marqo 두 모델은 색상을 명시적 학습 신호로 썼다고 밝히고 있다** — 그래서 여기서도 안
오르면 "체크포인트가 색상을 안 배워서"가 아니라 "zero-shot 방식 자체의 한계"라는 쪽으로
결론이 기운다(= §9의 VLM 유지 / 지도학습 헤드 결론이 굳어진다).

In [ ]:
from evaluate import evaluate_color

COLOR_METRICS = ("primary_acc", "off_palette_rate")

color_preds = {}  # backend key -> list[ColorPrediction]
color_rows = []


def _failed_color_row(backend_key, cond, key, reason):
    row = _failed_row(key, reason, COLOR_METRICS)
    row.update({"model": backend_key, "white_balance": bool(cond), "backend": key})
    return row


for backend_key in MODELS:
    for cond in COLOR_CONDITIONS:
        key = f"{backend_key}{cond}"
        try:
            preds = evaluate_color(gt.samples, key)
        except Exception as exc:
            color_rows.append(
                _failed_color_row(backend_key, cond, key, f"{type(exc).__name__}: {exc}")
            )
            continue

        # §3과 같은 이유 — 로딩 실패는 예외가 아니라 "primary_acc=0.00"으로 위장한다.
        failure = all_failed(preds)
        if failure:
            color_rows.append(
                _failed_color_row(backend_key, cond, key, f"전 샘플 실패 — {failure}")
            )
            continue

        color_preds[key] = preds
        n = len(preds)
        color_rows.append(
            {
                "model": backend_key,
                "white_balance": bool(cond),
                "backend": key,
                "n": n,
                "primary_acc": sum(p.correct_primary for p in preds) / n,
                "off_palette_rate": sum(p.off_palette for p in preds) / n,
                "error_n": error_count(preds),
                "error": "",
            }
        )
        _c = color_rows[-1]
        print(
            f"[{key}] primary_acc={_c['primary_acc']:.2f}"
            + (f"  (실패 {_c['error_n']}행 포함)" if _c["error_n"] else "")
        )

color_df = pd.DataFrame(color_rows)
display(color_df)

if not color_df.empty:
    # 전처리 효과가 체크포인트마다 다른지 — §13/§17은 범용 CLIP 하나에서만 확인된 결과다.
    wb_effect = color_df.pivot_table(index="model", columns="white_balance", values="primary_acc")
    print("\n화이트밸런스 효과 (열: False=원본, True=_wb)")
    display(wb_effect)

### 4-1. 색상 혼동행렬

`REPORT.md` §17이 범용 CLIP에서 **GRAY/NAVY/SILVER 쏠림("끌개" 패턴)** 을 정량 확인했다.
패션 체크포인트에서도 같은 라벨로 쏠린다면 원인은 체크포인트가 아니라 방식 쪽이고,
쏠림이 사라졌다면 content-free 편향 보정(§17)을 패션 모델에도 붙여볼 근거가 된다.

In [ ]:
from evaluate import color_confusion_matrix

# 원본 조건만 본다(_wb까지 8개를 다 그리면 표가 너무 많다 — 필요하면 키를 바꿔 재실행).
for backend_key in MODELS:
    key = f"{backend_key}{COLOR_CONDITIONS[0]}"
    if key not in color_preds:
        continue
    cm_df = pd.DataFrame(color_confusion_matrix(color_preds[key])).fillna(0).astype(int).T
    cm_df = cm_df.loc[cm_df.sum(axis=1) > 0, cm_df.sum(axis=0) > 0]
    print(f"\n[{key}] 행=정답, 열=예측")
    display(cm_df)

## 5. 지연 실측

프로덕션 모델 선정 기준이라 같이 잰다 — 정확도가 비슷하면 더 빠른 쪽을 고른다.
`marqo_fashion_siglip`은 ViT-B/16 기반이라 ViT-B/32인 나머지보다 느릴 것으로 예상된다
(패치가 작을수록 토큰 수가 늘어난다).

`latency.run_latency_suite()`는 `template_key`를 넘겨받지 않아 내부 기본값 `"kr_plain"`
으로 고정된다 — 정확도 측정과 같은 프롬프트로 재려고 `benchmark_backend()`를 직접
호출한다. 이 셀은 이미지 전체를 한 번 더 돌므로(모델도 다시 로드한다) 시간이 든다.

In [ ]:
latency_rows = []
if RUN_LATENCY:
    from latency import benchmark_backend

    for backend_key in MODELS:
        if backend_key not in category_preds:
            continue  # §3에서 이미 로드 실패한 모델은 다시 시도하지 않는다
        result = benchmark_backend(
            backend_key, real_images, template_key=TEMPLATE_KEY, stress="idle"
        )
        stats = result.stats()
        latency_rows.append(
            {
                "model": backend_key,
                "load_s": round(result.load_time_s, 2),
                "p50_ms": round(stats.get("p50", float("nan")), 1),
                "p95_ms": round(stats.get("p95", float("nan")), 1),
                "mean_ms": round(stats.get("mean", float("nan")), 1),
                "peak_mem_mb": round(result.peak_mem_mb, 1),
            }
        )
        _r = latency_rows[-1]
        print(f"[{backend_key}] p50={_r['p50_ms']}ms p95={_r['p95_ms']}ms load={_r['load_s']}s")

    display(pd.DataFrame(latency_rows))
else:
    print("RUN_LATENCY=False — §2 셀에서 True로 바꾸고 재실행할 것.")

## 6. 종합 스코어보드

모델당 한 줄. `_Δ` 컬럼은 범용 baseline(`clip_vit_b32`) 대비 차이다 — 이 노트북의 질문이
"패션 파인튜닝이 실제로 움직였는가"라서, 절대값보다 이 델타가 결론에 가깝다.

In [ ]:
_latency_by_model = {r["model"]: r for r in latency_rows}
_color_by_key = {r["backend"]: r for r in color_rows}

board_rows = []
for row in category_rows:
    model = row["model"]
    plain = _color_by_key.get(model, {})
    wb = _color_by_key.get(f"{model}_wb", {})
    lat = _latency_by_model.get(model, {})
    board_rows.append(
        {
            "model": model,
            "sub_top1": row["sub_top1"],
            "sub_top3": row["sub_top3"],
            "major_inverted": row["major_inverted"],
            "color_acc": plain.get("primary_acc", float("nan")),
            "color_acc_wb": wb.get("primary_acc", float("nan")),
            "p50_ms": lat.get("p50_ms", float("nan")),
            "load_s": lat.get("load_s", float("nan")),
            "note": row["error"],
        }
    )

board = pd.DataFrame(board_rows).set_index("model")

METRIC_COLS = ["sub_top1", "sub_top3", "major_inverted", "color_acc", "color_acc_wb"]
if BASELINE in board.index:
    delta = (board[METRIC_COLS] - board.loc[BASELINE, METRIC_COLS]).add_suffix("_Δ")
    board = board.join(delta)

display(board)
print("sub_top1/sub_top3 = 중분류 30-way, major_inverted = 그 예측을 대분류로 역산")
print(f"N={len(real_samples)} — MIN_N_FOR_CONFIDENCE(30) 미만이다. 소수점 차이는 과신하지 말 것.")

## 7. 리포트 저장

`build_report()`가 모델별 ECE·신뢰도 bin·임계값 추천·혼동행렬·대분류별 정확도까지 묶어
JSON으로 떨군다 — 노트북 출력은 재실행하면 사라지므로 숫자를 인용할 땐 이 파일을 쓴다.

In [ ]:
from evaluate import build_report, save_report

if category_preds or color_preds:
    report = build_report(gt, category_preds, color_preds)
    path = save_report(report, name="report_fashion_compare.json")
    print(f"리포트 저장: {path}")
    print(f"  category_runs: {list(report['category_runs'])}")
    print(f"  color_runs: {list(report['color_runs'])}")
else:
    print("저장할 결과가 없다 — §3/§4를 먼저 실행할 것.")

## 참고

- **이 노트북에서 나온 결론은 `REPORT.md` §18에 반영할 것** — 노트북 출력은 재실행하면
  사라진다. 저장된 JSON은 `outputs/report_fashion_compare.json`.
- 백엔드 정의와 후보 선정 근거: `classify.CATEGORY_MODELS` / `CATEGORY_MODEL_LOAD_OPTIONS`
  주석, `color.COLOR_BACKENDS` (g)항목.
- 대분류 6-way **직접** 질의는 여기서 안 잰다 — `REPORT.md` §14에서 범용 CLIP이 0.35로
  무너지고 17장 중 12장이 OUTER로 쏠리는 게 이미 확인됐고, 역산 방식(1.00)이 같은 비용에
  더 정확해서 직접 질의를 쓸 이유가 없다.
- VLM(Qwen3-VL) 대조군과 색상 개선 기법(프롬프트/스와치/편향 보정) 실측은 이 노트북이
  아니라 `major_category_and_color_check.ipynb`에 있다.